# Python Interview Preparation Guide
## Descriptors, Metaclasses & Context Managers

**Target**: Master these 3 advanced concepts quickly for Python interviews

---

### 📌 What You'll Learn

**Advanced Python** (6 concepts):
1. **Descriptors** - Control attribute access
2. **Metaclasses** - Create classes that create classes
3. **Context Managers** - Manage resources safely
4. **Decorators** - Enhance functions/methods
5. **Generators** - Create iterators efficiently
6. **Closures** - Capture variables in nested functions

**Python Fundamentals** (12 concepts):
7. Why is Python an Interpreted Language?
8. What is the GIL?
9. Mutable vs Immutable Objects
10. Pass by Reference or Value?
11. Duck Typing
12. *args and **kwargs
13. Generators vs Lists
14. List Comprehensions
15. Shallow Copy vs Deep Copy
16. `is` vs `==`
17. Lambda Functions
18. Variable Scope (Global, Local, Nonlocal)

Let's dive deep with theory + practical code! 🚀

# 1. DESCRIPTORS

## Theory: What is a Descriptor?

**Definition**: A descriptor is a Python object that implements `__get__`, `__set__`, and/or `__delete__` methods. It allows you to customize attribute access.

### Key Points:
- **Descriptor Protocol**: Objects can control how attributes are accessed, modified, or deleted
- **Types**:
  - **Data Descriptor**: Has `__set__` and/or `__delete__` (takes precedence)
  - **Non-Data Descriptor**: Has only `__get__`
- **Use Cases**: Properties, validators, computed attributes, lazy loading

### Method Signatures:
```python
def __get__(self, obj, objtype=None):
    # Called when attribute is accessed
    return value

def __set__(self, obj, value):
    # Called when attribute is assigned
    pass

def __delete__(self, obj):
    # Called when attribute is deleted
    pass
```

### How it Works:
1. When you access `obj.attr`, Python checks if `attr` is a descriptor
2. If it is, it calls the descriptor's `__get__` method instead of returning the value directly

In [ ]:
class IntValidator:
    def __init__(self, name):
        self.name = name
    def __get__(self, obj, objtype=None):
        return obj.__dict__.get(self.name)
    def __set__(self, obj, value):
        if not isinstance(value, int):
            raise TypeError(f"{self.name} must be int")
        obj.__dict__[self.name] = value

class Student:
    age = IntValidator("age")
    def __init__(self, age):
        self.age = age

s = Student(25)
print(f"Age: {s.age}")  # Descriptor __get__
s.age = 30
try:
    s.age = "invalid"
except TypeError as e:
    print(f"Error: {e}")

In [ ]:
class Temp:
    def __init__(self, celsius):
        self._c = celsius
    @property
    def celsius(self):
        return self._c
    @celsius.setter
    def celsius(self, value):
        if value < -273:
            raise ValueError("Too low")
        self._c = value
    @property
    def fahrenheit(self):
        return (self._c * 9/5) + 32

t = Temp(0)
print(f"0°C = {t.fahrenheit}°F")
t.celsius = 100
print(f"100°C = {t.fahrenheit}°F")

---

# 2. METACLASSES

## Theory: What is a Metaclass?

**Definition**: A metaclass is a "class of a class" - it defines how a class behaves. Just as an object is an instance of a class, a class is an instance of a metaclass.

### Key Points:
- **All classes are instances of `type`** (the default metaclass)
- **Metaclasses use `__new__` and `__init__`** to customize class creation
- **When do they execute?** When the class is defined, not when an instance is created
- **Use Cases**: 
  - Enforce naming conventions
  - Auto-register classes
  - Add methods automatically
  - Create singleton classes
  - API frameworks (Django models, SQLAlchemy)

### How it Works:
```python
# Step 1: Define a metaclass
class MyMeta(type):
    def __new__(mcs, name, bases, namespace):
        # Called when class is being created
        return super().__new__(mcs, name, bases, namespace)

# Step 2: Use it in class definition
class MyClass(metaclass=MyMeta):
    pass
```

In [ ]:
class Animal:
    def speak(self):
        return "Sound"

print(f"Type of Animal: {type(Animal)}")
print(f"Type of instance: {type(Animal())}")

# Create class dynamically
Dog = type('Dog', (Animal,), {'speak': lambda self: "Woof!"})
print(f"Dog says: {Dog().speak()}")

In [ ]:
class LogMeta(type):
    def __new__(mcs, name, bases, namespace):
        for attr, value in namespace.items():
            if callable(value) and not attr.startswith('_'):
                namespace[attr] = lambda *args, v=value, a=attr: (print(f"Calling {a}"), v(*args))[1]
        return super().__new__(mcs, name, bases, namespace)

class Calc(metaclass=LogMeta):
    def add(self, a, b):
        return a + b

c = Calc()
print(c.add(5, 3))

In [ ]:
class SingletonMeta(type):
    _instances = {}
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class DB(metaclass=SingletonMeta):
    def __init__(self, conn):
        self.conn = conn

db1 = DB("localhost")
db2 = DB("different")  # Reuses db1
print(f"Same instance? {db1 is db2}")  # True!

---

# 3. CONTEXT MANAGERS

## Theory: What is a Context Manager?

**Definition**: A context manager is an object that defines what happens when you enter and exit a `with` block.

### Key Points:
- **Protocol**: Implements `__enter__` and `__exit__` methods
- **`__enter__`**: Executed when entering the `with` block
- **`__exit__`**: Executed when exiting the block (even if an exception occurs!)
- **Exception Handling**: `__exit__` receives exception info if one occurred
- **Use Cases**:
  - File handling (automatic close)
  - Database connections
  - Locking mechanisms
  - Temporary state management
  - Resource cleanup

### Syntax:
```python
with expression as variable:
    # __enter__ called here
    # ... code block ...
# __exit__ called here (even if exception occurred)
```

### __exit__ Return Value:
- **Return `True`**: Suppress the exception
- **Return `False` or `None`**: Let the exception propagate

In [ ]:
class FileHandler:
    def __init__(self, filename, mode='r'):
        self.filename = filename
        self.mode = mode
    def __enter__(self):
        self.file = open(self.filename, self.mode)
        return self.file
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.file.close()

with open('/tmp/test.txt', 'w') as f:
    f.write("Hello!")

with FileHandler('/tmp/test.txt') as f:
    print(f"Content: {f.read()}")

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timer(name):
    start = time.time()
    print(f"Starting {name}")
    try:
        yield
    finally:
        print(f"Done in {time.time()-start:.2f}s")

with timer("Task"):
    time.sleep(0.1)
    print("Working...")

In [ ]:
class DB:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"Connected to {self.name}")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Disconnected")
    def query(self, sql):
        return f"Result: {sql}"

with DB("mydb") as db:
    print(db.query("SELECT *"))

---

# 🎓 INTERVIEW Q&A QUICK REFERENCE

## Common Questions & Answers

### Descriptors
**Q: What's the difference between `@property` and a regular descriptor?**
- A: `@property` is a built-in descriptor. Regular descriptors are more flexible - you can apply them to multiple attributes.

**Q: What's a data descriptor vs non-data descriptor?**
- A: Data descriptors have `__set__` (take precedence). Non-data descriptors only have `__get__`. Data descriptors override instance dict.

**Q: How does attribute lookup work?**
- A: Data descriptor → Instance dict → Non-data descriptor → Class dict → Parent classes

---

### Metaclasses
**Q: When do metaclasses execute?**
- A: When the class is DEFINED, not when instances are created.

**Q: What's the difference between `__new__` and `__init__` in metaclasses?**
- A: `__new__` creates the class object itself. `__init__` initializes the class after creation.

**Q: Real-world example of metaclasses?**
- A: Django models, SQLAlchemy ORM, ABCs (Abstract Base Classes)

---

### Context Managers
**Q: What happens if an exception occurs in a `with` block?**
- A: `__exit__` is ALWAYS called. It receives exception info and can suppress it by returning `True`.

**Q: How do you write a context manager with `@contextmanager`?**
- A: Use the decorator. Code before `yield` = `__enter__`. Code after = `__exit__`.

**Q: When should I use context managers?**
- A: Whenever you need guaranteed cleanup: files, connections, locks, temporary state

---

# 📝 SUMMARY & CHEAT SHEET

| Concept | When Executes | Use Case | Key Methods |
|---------|------|----------|-------------|
| **Descriptor** | Attribute access | Validation, computed properties | `__get__`, `__set__`, `__delete__` |
| **Metaclass** | Class definition | Enforce conventions, auto-register | `__new__`, `__init__` |
| **Context Manager** | `with` statement | Resource cleanup, guaranteed finalization | `__enter__`, `__exit__` |

---

## Quick Tips for Interviews

✅ **DO:**
- Show understanding of when each concept executes
- Provide real-world examples (Django, SQLAlchemy)
- Explain method resolution order (MRO)
- Mention exception handling in context managers
- Give practical, runnable code examples

❌ **DON'T:**
- Confuse descriptors with properties (property is just one type)
- Think metaclasses run at instance creation
- Forget that `__exit__` always runs even with exceptions
- Over-engineer solutions - use built-ins when possible

---

## Practice Problems

1. **Create a descriptor** that converts strings to uppercase automatically
2. **Write a metaclass** that validates all class methods have docstrings
3. **Build a context manager** that temporarily changes a variable and restores it
4. **Combine all three**: Metaclass that adds descriptor with context manager support

Good luck with your Python interview! 🚀

# 4. BASIC PYTHON FUNDAMENTALS

Quick reference for basic but crucial Python concepts. Each concept has a short code example (1-10 lines).

## 1. Why is Python an Interpreted Language?

**Answer**: Python code is executed line-by-line by an interpreter (CPython) rather than compiled to machine code first. Each statement is translated to bytecode at runtime.

**Key Point**: This makes Python slower but more flexible and easier to debug.

In [ ]:
x = 5  # Interpreted line by line
y = 10
print(x + y)  # Each line executed immediately
# Error on line 5 won't stop lines 1-3 from running

---

## 2. What is the GIL (Global Interpreter Lock)?

**Answer**: The GIL is a mutex that protects access to Python objects in CPython. Only one thread can execute Python bytecode at a time, even on multi-core systems.

**Impact**: Makes multi-threading ineffective for CPU-bound tasks (use multiprocessing instead).

In [ ]:
import threading
def task():
    return sum(range(10000000))

t1 = threading.Thread(target=task)
t2 = threading.Thread(target=task)
t1.start()
t2.start()
print("GIL prevents true parallelism in Python")

---

## 3. Mutable vs Immutable

**Mutable**: Can be changed after creation (list, dict, set)
**Immutable**: Cannot be changed (int, float, str, tuple)

**Interview Tip**: Immutables can be dict keys; mutables cannot.

In [ ]:
# Tuple (immutable) as dict key
d = {(1, 2): "value"}
print(d)  # Works

# List (mutable) as key - ERROR
try:
    d = {[1, 2]: "value"}
except TypeError:
    print("Lists can't be dict keys (mutable)")

---

## 4. Pass by Value vs Pass by Reference (Object References)

**Answer**: Python uses "pass by object reference" - you're passing a reference to the object, not a copy.

In [ ]:
def modify(lst):
    lst.append(4)  # Modifies original

my_list = [1, 2, 3]
modify(my_list)
print(my_list)  # [1, 2, 3, 4]

def change(x):
    x = 99  # Doesn't affect original
x = 5
change(x)
print(x)  # Still 5

---

## 5. Duck Typing

**Answer**: "If it walks like a duck and quacks like a duck, it's a duck" - Python doesn't care about type, only behavior.

In [ ]:
class Dog:
    def speak(self):
        return "Woof!"

class Cat:
    def speak(self):
        return "Meow!"

def sound(animal):
    print(animal.speak())

sound(Dog())   # Works
sound(Cat())   # Also works - no type check needed

---

## 6. *args and **kwargs

***args**: Accept variable number of positional arguments (tuple)
****kwargs**: Accept variable number of keyword arguments (dict)

In [ ]:
def func(*args, **kwargs):
    print(f"args: {args}")
    print(f"kwargs: {kwargs}")

func(1, 2, 3, name="Alice", age=25)

---

## 7. Generators vs Lists

**Generators**: Lazy evaluation, memory efficient, use `yield`
**Lists**: Eager evaluation, stores all items in memory

In [ ]:
# List (eager) - creates all at once
lst = [x**2 for x in range(5)]
print(f"List: {lst}")

# Generator (lazy) - creates on demand
gen = (x**2 for x in range(5))
print(f"Generator: {list(gen)}")

---

## 8. List Comprehension

**Concise and efficient way to create lists**

In [ ]:
numbers = [1, 2, 3, 4, 5]

# Traditional loop
result1 = []
for n in numbers:
    if n % 2 == 0:
        result1.append(n ** 2)

# List comprehension (better)
result2 = [n**2 for n in numbers if n % 2 == 0]
print(result2)  # [4, 16]

---

## 9. Shallow Copy vs Deep Copy

**Shallow copy**: Copies references (nested objects still shared)
**Deep copy**: Copies everything recursively

In [ ]:
import copy

original = [[1, 2], [3, 4]]
shallow = copy.copy(original)
shallow[0][0] = 99
print(f"Shallow: original changed = {original[0][0] == 99}")  # True!

original = [[1, 2], [3, 4]]
deep = copy.deepcopy(original)
deep[0][0] = 99
print(f"Deep: original unchanged = {original[0][0] == 1}")  # True

---

## 10. is vs ==

**==**: Checks value equality
**is**: Checks object identity (same memory address)

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(f"a == b: {a == b}")  # True (value)
print(f"a is b: {a is b}")  # False (different objects)
print(f"a is c: {a is c}")  # True (same object)

---

## 11. Lambda Functions

**Anonymous function for simple, single-use operations**

In [ ]:
numbers = [1, 2, 3, 4, 5]

# Map
squared = list(map(lambda x: x**2, numbers))
print(squared)  # [1, 4, 9, 16, 25]

# Filter
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(evens)  # [2, 4]

---

## 12. Variable Scope (LEGB)

**L**ocal → **E**nclosing → **G**lobal → **B**uilt-in

In [ ]:
def outer():
    x = "Enclosing"
    def inner():
        x = "Local"
        print(x)
    inner()  # Local
    print(x)  # Enclosing
outer()

# Modify enclosing with 'nonlocal'
def counter():
    count = 0
    def inc():
        nonlocal count
        count += 1
        return count
    return inc
c = counter()
print(c(), c(), c())  # 1, 2, 3

---

# 13. CLOSURES (Important Interview Topic!)

## What is a Closure?

**Definition**: A closure is a function that "remembers" variables from its enclosing scope, even after the outer function has finished executing.

**Key Points**:
- Inner function captures variables from outer function
- Inner function can access and modify those variables
- The captured variables persist in memory
- Used for: Factory functions, callbacks, decorators, data hiding

**How it Works**:
1. Outer function defines a variable
2. Inner function references that variable
3. Outer function returns inner function
4. Inner function still has access to the variable!

In [ ]:
# Example 1: Basic Closure
def outer(x):
    print(f"Outer called with x={x}")
    def inner():
        print(f"Inner can access x={x}")
        return x * 2
    return inner

# inner function "remembers" x even after outer finishes
func = outer(5)
print(func())  # 10
print(func())  # 10 (x is still 5!)

# Proof: x is captured in closure
print(f"Closure variables: {func.__closure__[0].cell_contents}")  # 5

In [ ]:
# Example 2: Counter using Closure (Very Common Interview Question!)
def make_counter():
    count = 0  # Captured in closure
    def increment():
        nonlocal count  # Modify enclosing variable
        count += 1
        return count
    return increment

counter = make_counter()
print(counter())  # 1
print(counter())  # 2
print(counter())  # 3

# Each closure has its own captured state
counter2 = make_counter()
print(counter2())  # 1 (separate counter!)

In [ ]:
# Example 3: Private Variables using Closures (Data Hiding)
def create_account(initial_balance):
    balance = initial_balance  # Private variable!
    
    def deposit(amount):
        nonlocal balance
        balance += amount
        return f"Deposited {amount}, balance: {balance}"
    
    def withdraw(amount):
        nonlocal balance
        if amount > balance:
            return "Insufficient funds"
        balance -= amount
        return f"Withdrew {amount}, balance: {balance}"
    
    def get_balance():
        return balance
    
    return deposit, withdraw, get_balance

deposit, withdraw, get_balance = create_account(1000)
print(deposit(500))      # Deposited 500
print(withdraw(200))     # Withdrew 200
print(get_balance())     # 1300

In [ ]:
# Example 4: Closure in Decorator Pattern
def log_calls(func):
    call_count = 0  # Captured in closure
    def wrapper(*args, **kwargs):
        nonlocal call_count
        call_count += 1
        print(f"Call #{call_count}: {func.__name__}{args}")
        return func(*args, **kwargs)
    return wrapper

@log_calls
def greet(name):
    return f"Hello {name}"

print(greet("Alice"))  # Call #1: greet('Alice')
print(greet("Bob"))    # Call #2: greet('Bob')

---

## Closure vs Global Variable

**Closure**: Encapsulated, private, each instance has its own state
**Global**: Public, shared across entire program (BAD practice!)

**Example**:

In [ ]:
# BAD: Using Global Variable
count_global = 0
def increment_global():
    global count_global
    count_global += 1
    return count_global

# GOOD: Using Closure
def make_counter_good():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

print("Global approach (shared):")
print(increment_global())  # 1
print(increment_global())  # 2

print("\nClosure approach (private):")
c1 = make_counter_good()
c2 = make_counter_good()
print(c1())  # 1
print(c1())  # 2
print(c2())  # 1 (separate!)

---

## Key Interview Points on Closures

✅ **What**: Function that captures variables from outer scope  
✅ **Why**: Data encapsulation, private variables, stateful functions  
✅ **When**: Decorators, callbacks, factory functions, counters  
✅ **How**: Use `nonlocal` to modify captured variables  

**Interview Question Examples**:
1. "What's the difference between closure and global?" → Encapsulation!
2. "How do you create private variables in Python?" → Use closures!
3. "Explain decorators" → Closures + functions!
4. "How does `nonlocal` work?" → Modifies captured variable in outer scope!

---

# 14. DECORATORS

**Definition**: A function that takes another function and extends/modifies its behavior without permanently changing it.

**Key Point**: Decorators = Closures + Wrapper functions

### How to Recognize Decorators:
```python
@decorator_name
def my_function():
    pass
```

In [ ]:
# Example 1: Basic Decorator
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("Before function call")
        result = func(*args, **kwargs)
        print("After function call")
        return result
    return wrapper

@my_decorator
def say_hello(name):
    return f"Hello {name}"

say_hello("Alice")  # Prints: Before, Hello Alice, After

In [ ]:
# Example 2: Decorator with Arguments
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(3)
def greet():
    print("Hi!")

greet()  # Prints "Hi!" 3 times

In [ ]:
# Example 3: Timing Decorator
import time
def timer_decorator(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"Took {time.time()-start:.4f}s")
        return result
    return wrapper

@timer_decorator
def slow_function():
    time.sleep(0.1)
    return "Done"

slow_function()  # Shows execution time

---

# 15. GENERATORS

**Definition**: A generator is a function that yields values one at a time, pausing execution between yields.

**Key Points**:
- Use `yield` keyword instead of `return`
- Creates iterator objects
- Memory efficient (lazy evaluation)
- Use: Large data processing, infinite sequences

In [ ]:
# Example 1: Simple Generator
def count_up(n):
    i = 0
    while i < n:
        yield i  # Pauses here, resumes next call
        i += 1

gen = count_up(3)
print(next(gen))  # 0
print(next(gen))  # 1
print(next(gen))  # 2
# next(gen) would raise StopIteration

In [ ]:
# Example 2: Infinite Generator
def infinite_sequence():
    num = 0
    while True:
        yield num
        num += 1

gen = infinite_sequence()
print(next(gen))  # 0
print(next(gen))  # 1
print(next(gen))  # 2
# Can keep going forever - memory efficient!

In [ ]:
# Example 3: Generator Expression (Like List Comprehension)
squares = (x**2 for x in range(5))  # () not []
print(list(squares))  # [0, 1, 4, 9, 16]

# Generator vs List
gen_memory = (x for x in range(1000000))  # Small memory
list_memory = [x for x in range(1000000)]  # Large memory!

# Use with for loop
for value in (x**2 for x in range(3)):
    print(value)  # 0, 1, 4